# NotebookLM Kit — Video Pipeline

Generates one video per source, downloads all as MP4.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Template** — choose format, visual style, and optional focus prompt
4. **List Sources** — preview sources; pick indices to filter
5. **Create** — submit one video job per source; jobs saved to disk
6. **Poll** — wait for generation, or resume from a previous run
7. **Download** — save each video as `<timestamp>_<Notebook>__<Source>.mp4`


In [ ]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT

# First time only — opens a browser window for Google login, saves credentials.json:
#   login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials ready — token: 42 chars, cookies: 2316 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
NOTEBOOK_ID = "64d7d18f-a2db-4b0b-959d-5276506dffad"  # ← paste your notebook ID here

## List Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources (title, status, full ID).  
Filter by index before passing to **Create**:  `sources[:1]` to test one first  /  `[sources[i] for i in [0, 2]]`


In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)



Notebook : Mastering Modern Data Orchestration with Dagster
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| # | Title                                                                    | Status     | Source ID                            |
+---+--------------------------------------------------------------------------+------------+--------------------------------------+
| 0 | A Quick Introduction to Machine Learning with Dagster | HackerNoon       | READY      | fbcb2f70-5f48-4a30-93ce-6099e82a0cd5 |
| 1 | Best Practices for Machine Learning with Dagster                         | READY      | e06a6437-95a0-4bb0-b886-66fcb4e3b57a |
| 2 | Dagster for Data Orchestration: Architecture & Capabilities - Atlan      | READY      | 7b94d750-50a8-4d6c-a059-9fdc8d2bd502 |
| 3 | Dagster training: Orchestrating data pipelines in the cloud - Ambient IT | READY      | bf05939f-1b17-44f5-a64e-6a2a83e5d514 |
| 4 | Da

## Video Template

Set format, visual style, and an optional focus prompt.


In [ ]:
from config import VIDEO_PROMPT

VIDEO_INSTRUCTIONS = VIDEO_PROMPT  # optional focus / style prompt — leave empty for default

# FORMAT:       1 = Explainer   2 = Brief      3 = Cinematic  (Cinematic ignores visualStyle)
# VISUAL STYLE: 1 = Auto        2 = Classic    3 = Whiteboard   4 = Heritage  9 = Kawaii
VIDEO_CUSTOMIZATION = {
    "format":      1,
    "visualStyle": 1,
    "focus":       VIDEO_INSTRUCTIONS,
    # "language": "en",  # BCP-47 — omit to use notebook default
}


## Create Videos

`create_artifacts(notebook_id, type, sources, customization, instructions, creds)`

- Jobs saved to `jobs/video/<timestamp>_<NotebookName>.json`
- `instructions` — additional prompt; pass `None` to omit
- Skip this cell and run **Resume** below if jobs were already submitted


In [ ]:
from pipeline import create_artifacts

video_jobs = create_artifacts(
    NOTEBOOK_ID, "VIDEO", sources[1],
    VIDEO_CUSTOMIZATION, VIDEO_INSTRUCTIONS or None,
    creds,
)

# sources = sources[:1]                   # test with one first
# sources = [sources[i] for i in [0, 2]]  # specific picks


Submitted 1 job(s)  →  jobs\video\20260530_090711_Mastering_Modern_Data_Orchestr.json
+---+--------------------------------------------------+----------------------------------------------+
| # | Source                                           | Artifact ID                                  |
+---+--------------------------------------------------+----------------------------------------------+
| 0 | Best Practices for Machine Learning with Dagster | e5302356-b2c7-4e6a-b8de-096242f3bb91         |
+---+--------------------------------------------------+----------------------------------------------+


## Poll Until Ready

`poll_jobs(jobs, notebook_id, creds, *, interval=60, max_wait_min=30)`

- `interval` — seconds between checks (videos take longer; 60s recommended)
- `max_wait_min` — timeout; returns `False` if exceeded (safe to re-run)

**Resume** (cell below): run instead of **Create** when continuing a previous session.


In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ───
from pipeline import load_jobs
video_jobs = load_jobs("VIDEO")


Loaded 1 job(s) from 20260530_090711_Mastering_Modern_Data_Orchestr.json
+---+--------------------------------------------------+----------------------------------------------+
| # | Source                                           | Artifact ID                                  |
+---+--------------------------------------------------+----------------------------------------------+
| 0 | Best Practices for Machine Learning with Dagster | e5302356-b2c7-4e6a-b8de-096242f3bb91         |
+---+--------------------------------------------------+----------------------------------------------+


In [ ]:
from pipeline import poll_jobs

poll_jobs(video_jobs, NOTEBOOK_ID, creds, interval=60, max_wait_min=60)

  ✓ Best Practices for Machine Learning with Dagster        READY

✅ All artifacts ready — proceed to download.


True

## Download

Each video is saved as `<source title>.mp4` in `outputs/video/`.

In [ ]:
from pipeline import download_artifacts

download_artifacts(video_jobs, NOTEBOOK_ID, "VIDEO", creds)

  Resolving video URL for: Best Practices for Machine Learning with Dagster …

Downloaded 0 / 1 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\video
+---+--------+----------------------------------------------+------------+
| # | Source | File                                         | Status     |
+---+--------+----------------------------------------------+------------+
+---+--------+----------------------------------------------+------------+
  x  Best Practices for Machine Learning with Dagster: asyncio.run() cannot be called from a running event loop
+---+--------+----------------------------------------------+------------+


d:\Core\_Code D\notebooklm-kit\pipeline\_artifacts.py:443: RuntimeWarning: coroutine '_resolve_and_download_video' was never awaited


[]